# 🪷 隆波帕默尊者法谈 · Google Colab GPU 全自动文字化、结构化与泰汉字幕工作流

专为 **Google One AI Pro + Colab GPU (T4 / L4 / A100) + Google Drive 2TB** 设计的全自动化法音整理流水线：

### ✨ 解决痛点与核心能力
1. **免本地算力 & 持久化存档**：所有音频、原始字幕、结构化 JSON 与 Markdown 全部直接存入你的 `Google Drive/Luangpor_Dhamma_Archive`，支持断点续传，不占电脑硬盘。
2. **双渠道稳健获取音源与官方字幕**：
   - **优先官方纯净 MP3**（`media.dhamma.com`，无 YouTube 机房 IP 风控、音质极净、自动跳过直播前空等静音）；
   - **YouTube 官方字幕直取**（若官方已出手动泰语 CC，1 秒无损抓取，直接跳过语音识别）；
   - **支持 YouTube Cookies + 批量目录队列**：自动读取本站固化的年度目录，一键列出尚未制作字幕的场次批量排队处理。
3. **Colab GPU 高精度泰语语音识别 (ASR)**：
   - 支持 **`biodatlab/whisper-th-large-v3-combined`**（泰国玛希隆大学微调泰语专属大模型）与 **`faster-whisper large-v3`**（FP16 + Silero VAD 静音过滤）；
   - 注入隆波帕默尊者**高频巴利-泰语佛法词汇引导词表**，并按自然停顿自动切分为 4~10 秒短句（最适合卡拉 OK 逐词跟读）。
4. **结构化与泰译中（双模式任选）**：
   - **模式 A（全自动 API 批处理）**：调用现代 `google-genai` SDK（`gemini-2.5-pro` / `gemini-2.5-flash`），内置尊者法谈词典校对错字 + 泰译中 + 精细分词 + **朱拉隆功五声调 Adapted IPA**。
   - **模式 B（零 API 费用 · Google One Pro 网页版 Gemini 接力）**：自动生成分段好的标准 Prompt 文本包，直接复制进你现有的 Gemini 网页版运行并自动回填合并。
5. **一键产出三大结构化成果**：
   - `*.enhanced.srt` — 可直接拖入**泰法语声闻共学网站**（「共学互助字幕校对与上传」或「课件导入工作台」）立即播放；
   - `*.taixue.json` — 完整 `SubtitleLine[]` 结构化数据集（也可直接发给官网固化上线）；
   - `*.md` — 泰汉双语法谈实录文档（直接上传到 Google One Pro 自带的 **NotebookLM Plus**，建立全库 AI 法义检索知识库）。

> **运行前请确认**：顶部菜单 **代码执行程序 (Runtime) → 更改运行时类型 (Change runtime type) → 硬件加速器选择 `T4 GPU` 或 `L4 GPU`**。

## 1️⃣ 挂载 Google Drive 并初始化持久化归档目录

In [ ]:
#@title 挂载 Google Drive & 创建归档目录结构
import os, json
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

BASE_DIR = Path('/content/drive/MyDrive/Luangpor_Dhamma_Archive')
DIRS = {
    'audio': BASE_DIR / '01_raw_audio',
    'raw_srt': BASE_DIR / '02_raw_subtitles',
    'json': BASE_DIR / '03_structured_json',
    'enhanced_srt': BASE_DIR / '04_enhanced_srt',
    'markdown': BASE_DIR / '05_markdown_kb',
    'chunks': BASE_DIR / '06_web_prompt_chunks',
    'manifests': BASE_DIR / 'manifests',
}
for p in DIRS.values():
    p.mkdir(parents=True, exist_ok=True)

MANIFEST_PATH = DIRS['manifests'] / 'manifest.json'
if not MANIFEST_PATH.exists():
    MANIFEST_PATH.write_text(json.dumps({}, ensure_ascii=False, indent=2), encoding='utf-8')

print('✅ Google Drive 归档目录已就绪：', BASE_DIR)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "⚠️ 当前未检测到 GPU，请去 Runtime 切换为 T4/L4 GPU"

## 2️⃣ 安装依赖环境（yt-dlp + Whisper GPU + PyThaiNLP + Google GenAI）

In [ ]:
#@title 一键安装所需工具与泰语处理库（约 1-2 分钟）
!pip install -q -U yt-dlp faster-whisper ctranslate2 transformers accelerate pythainlp python-crfsuite google-genai requests tqdm
!apt-get -qq update && apt-get -qq install -y ffmpeg > /dev/null

import torch
print('PyTorch:', torch.__version__, '| CUDA 可用:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU 型号:', torch.cuda.get_device_name(0))

## 3️⃣ 选择任务源（支持单场直链、YouTube 视频或官网年度目录批量拉取）

- **最稳推荐**：填入 `dhamma.com` 公开 MP3（如 `https://media.dhamma.com/pramote/cd/2569/691025.mp3`），**完全不受 YouTube 机房反爬限制**。
- **如果处理 YouTube**：若遇到 Colab IP 提示 `Sign in to confirm you're not a bot`，只需在本地 Chrome 安装 *Get cookies.txt LOCALLY* 插件导出 `cookies.txt` 并上传到 `MyDrive/Luangpor_Dhamma_Archive/manifests/cookies.txt` 即可。

In [ ]:
#@title 3.1 配置单场任务参数
import re, requests

TASK_ID = "2026-10-25"  #@param {type:"string"}
TALK_TITLE_TH = "2026-10-25 หลวงปู่ปราโมทย์ ปาโมชฺโช"  #@param {type:"string"}
TALK_TITLE_ZH = "2026年10月25日 隆波帕默尊者开示"  #@param {type:"string"}

# 优先填 dhamma.com MP3 直链（最稳）；或填 YouTube URL；两者也可同时填（优先抓 YT 官方 CC，无 CC 则用 MP3 跑 Whisper）
DHAMMA_MP3_URL = "https://media.dhamma.com/pramote/cd/2569/691025.mp3"  #@param {type:"string"}
YOUTUBE_URL = ""  #@param {type:"string"}

# 可选：若抓 YouTube 时遇到 Bot 验证，把导出的 cookies.txt 放到 MyDrive/Luangpor_Dhamma_Archive/manifests/cookies.txt
COOKIES_PATH = DIRS['manifests'] / 'cookies.txt'

# ASR 识别引擎选择：
# - "faster-whisper-large-v3": 速度极快 + Silero VAD 自动去静音 + 词级时间戳精确
# - "biodatlab-whisper-th-large-v3": 泰国玛希隆大学微调泰语专属大模型，对泰语语音识别极准
ASR_ENGINE = "faster-whisper-large-v3"  #@param ["faster-whisper-large-v3", "biodatlab-whisper-th-large-v3"]

print(f"📌 当前任务: {TASK_ID} | 引擎: {ASR_ENGINE}")
print(f"   MP3 源: {DHAMMA_MP3_URL or '(未填写)'}")
print(f"   YouTube 源: {YOUTUBE_URL or '(未填写)'}")

In [ ]:
#@title 3.2 （可选）从网站在线目录拉取【待制作字幕】的场次清单
#@markdown 运行本单元可自动拉取本站的 2026 年度音视频目录，查看哪些日期已有泰语 MP3/视频但尚未制作精校字幕，方便按批次推进。
SITE_BASE_URL = ""  #@param {type:"string"}
# 例如填入你的网站 URL，留空则跳过

if SITE_BASE_URL.strip():
    base = SITE_BASE_URL.strip().rstrip('/')
    try:
        r_yt = requests.get(f"{base}/data/luangpor-youtube-catalog.json", timeout=15).json()
        todo = [
            it for it in r_yt.get('items', [])
            if not it.get('matchedLessonId') and not it.get('isRerun') and (it.get('thai') or {}).get('url')
        ]
        print(f"📋 发现 {len(todo)} 场尚未匹配课件的泰语开示（显示最近 15 场）：")
        for it in todo[:15]:
            cc = it.get('thaiCaptionStatus', 'unknown')
            print(f"  - {it['date']} | CC状态: {cc:18s} | {it['thai']['url']} | {it['thai']['title']}")
    except Exception as e:
        print("⚠️ 获取在线目录失败：", e)
else:
    print("ℹ️ 未填写 SITE_BASE_URL，将直接处理上方 3.1 设定的单场任务。")

## 4️⃣ 抓取官方字幕（无损直取）或下载纯音频跑 GPU 语音转文字 (Whisper)

In [ ]:
#@title 4.1 优先尝试获取 YouTube 官方泰语字幕 / 下载音频到 Google Drive
import subprocess, shutil

audio_file = DIRS['audio'] / f"{TASK_ID}.mp3"
raw_srt_file = DIRS['raw_srt'] / f"{TASK_ID}_raw_th.srt"
official_cc_found = False

def run_cmd(cmd):
    print("▶", " ".join(cmd))
    return subprocess.run(cmd, text=True, capture_output=True)

cookie_args = ["--cookies", str(COOKIES_PATH)] if COOKIES_PATH.exists() else []

# 1) 若提供了 YouTube 链接，优先尝试抓取官方人工泰语字幕 (sub-langs th)
if YOUTUBE_URL.strip() and not raw_srt_file.exists():
    tmp_sub_dir = Path('/content/tmp_subs')
    tmp_sub_dir.mkdir(exist_ok=True)
    res = run_cmd([
        "yt-dlp", *cookie_args,
        "--skip-download", "--write-subs", "--sub-langs", "th",
        "--convert-subs", "srt",
        "-o", str(tmp_sub_dir / f"{TASK_ID}.%(ext)s"),
        YOUTUBE_URL.strip()
    ])
    found_srts = list(tmp_sub_dir.glob(f"{TASK_ID}*.srt"))
    if found_srts:
        shutil.copy(found_srts[0], raw_srt_file)
        official_cc_found = True
        print(f"🎉 成功直接抓取到 YouTube 官方人工泰语字幕！已保存至：{raw_srt_file}（无需再跑 ASR！）")
    else:
        print("ℹ️ 该视频暂无官方手动泰语 CC 字幕，接下来将下载音频用 Colab GPU 识别。")

if raw_srt_file.exists():
    official_cc_found = True
    print(f"📄 已存在泰文 SRT：{raw_srt_file}")

# 2) 下载音频（优先 dhamma.com 纯净 MP3；否则从 YouTube 提取音频）
if not official_cc_found and not audio_file.exists():
    if DHAMMA_MP3_URL.strip():
        print("⬇️ 正在从 dhamma.com 下载官方纯净 MP3...")
        r = requests.get(DHAMMA_MP3_URL.strip(), stream=True, timeout=60)
        r.raise_for_status()
        with open(audio_file, 'wb') as f:
            for chunk in r.iter_content(chunk_size=1024*256):
                if chunk:
                    f.write(chunk)
        print(f"✅ MP3 下载完成：{audio_file} ({audio_file.stat().st_size / 1024 / 1024:.1f} MB)")
    elif YOUTUBE_URL.strip():
        print("⬇️ 正在从 YouTube 提取音轨 MP3...")
        res = run_cmd([
            "yt-dlp", *cookie_args,
            "-x", "--audio-format", "mp3", "--audio-quality", "128K",
            "-o", str(DIRS['audio'] / f"{TASK_ID}.%(ext)s"),
            YOUTUBE_URL.strip()
        ])
        if res.returncode != 0:
            print("❌ yt-dlp 下载失败：\n", res.stderr)
        else:
            print(f"✅ YouTube 音轨提取完成：{audio_file}")
    else:
        raise ValueError("请至少填写 DHAMMA_MP3_URL 或 YOUTUBE_URL 其中一项！")

In [ ]:
#@title 4.2 若无官方字幕，启动 Colab GPU 泰语语音识别并智能断句
import gc, torch

def fmt_srt_time(seconds: float) -> str:
    ms = int(round(max(0.0, seconds) * 1000))
    h, ms = divmod(ms, 3600_000)
    m, ms = divmod(ms, 60_000)
    s, ms = divmod(ms, 1000)
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"

def split_words_to_clauses(words, max_chars=75, max_dur=9.5, pause_gap=0.38):
    """按自然停顿间隙与最大长度将 Whisper 词级时间戳切分为适合卡拉 OK 跟读的短句"""
    if not words:
        return []
    clauses = []
    buf = []
    for w in words:
        if buf:
            gap = w['start'] - buf[-1]['end']
            cur_text = "".join(x['word'] for x in buf).strip()
            cur_dur = w['end'] - buf[0]['start']
            if gap >= pause_gap or len(cur_text) >= max_chars or cur_dur >= max_dur:
                clauses.append({
                    'start': buf[0]['start'],
                    'end': buf[-1]['end'],
                    'text': cur_text
                })
                buf = []
        buf.append(w)
    if buf:
        clauses.append({
            'start': buf[0]['start'],
            'end': buf[-1]['end'],
            'text': "".join(x['word'] for x in buf).strip()
        })
    return clauses

# 隆波帕默尊者常用佛法词表引导，显著提升泰语巴利词拼写准确率
DHAMMA_INITIAL_PROMPT = (
    "หลวงปู่ปราโมทย์ ปาโมชฺโช วัดสวนสันติธรรม แสดงธรรม "
    "สติ สมาธิ ปัญญา ไตรลักษณ์ อนิจจัง ทุกขัง อนัตตา "
    "อริยสัจ 4 อริยมรรคมีองค์ 8 กุศลกรรมบถ ศีล 5 จิตผู้รู้ "
    "รูปธรรม นามธรรม วิปัสสนา สมถะ วิมุตติ มรรคผล สังสารวัฏ หลวงปู่ดูลย์"
)

if raw_srt_file.exists():
    print(f"⏭️ 跳过 ASR：已存在泰文 SRT ({raw_srt_file})")
else:
    segments_out = []
    if ASR_ENGINE == "faster-whisper-large-v3":
        from faster_whisper import WhisperModel
        print("🚀 加载 faster-whisper large-v3 (GPU FP16 + Silero VAD)...")
        model = WhisperModel("large-v3", device="cuda", compute_type="float16")
        raw_segs, info = model.transcribe(
            str(audio_file),
            language="th",
            beam_size=5,
            vad_filter=True,
            vad_parameters=dict(min_silence_duration_ms=450),
            word_timestamps=True,
            initial_prompt=DHAMMA_INITIAL_PROMPT,
        )
        print(f"🎙️ 音频时长: {info.duration/60:.1f} 分钟，开始流式转写...")
        for seg in raw_segs:
            if seg.words:
                w_list = [{'start': w.start, 'end': w.end, 'word': w.word} for w in seg.words]
                for cl in split_words_to_clauses(w_list):
                    if cl['text']:
                        segments_out.append(cl)
            elif seg.text.strip():
                segments_out.append({'start': seg.start, 'end': seg.end, 'text': seg.text.strip()})
        del model
    else:
        from transformers import pipeline
        print("🚀 加载泰国玛希隆大学微调大模型 biodatlab/whisper-th-large-v3-combined...")
        asr = pipeline(
            task="automatic-speech-recognition",
            model="biodatlab/whisper-th-large-v3-combined",
            chunk_length_s=25,
            stride_length_s=4,
            device=0 if torch.cuda.is_available() else -1,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        )
        res = asr(str(audio_file), return_timestamps=True, generate_kwargs={"language": "thai", "task": "transcribe"})
        for ch in res.get("chunks", []):
            ts = ch.get("timestamp") or (0.0, 0.0)
            txt = (ch.get("text") or "").strip()
            if txt and ts[0] is not None:
                segments_out.append({
                    'start': float(ts[0]),
                    'end': float(ts[1] if ts[1] is not None else ts[0] + 4.0),
                    'text': txt
                })
        del asr

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # 写入标准原始泰文 SRT
    srt_lines = []
    for idx, seg in enumerate(segments_out, 1):
        srt_lines.append(f"{idx}\n{fmt_srt_time(seg['start'])} --> {fmt_srt_time(seg['end'])}\n{seg['text']}\n")
    raw_srt_file.write_text("\n".join(srt_lines), encoding="utf-8")
    print(f"✅ GPU 语音识别完成！共生成 {len(segments_out)} 句泰文字幕，已存至：{raw_srt_file}")

## 5️⃣ 结构化拆词、佛法校对与泰译中（双模式任选）

请根据你的使用习惯选择以下**任意一种方式**：
- **方式 A（全自动 Gemini API 通道）**：填入你的 Google AI Studio API Key，Colab 自动按每批 30 句调用 **`gemini-2.5-flash` / `gemini-2.5-pro`** 完成：**修正 ASR 错字 + 泰译中 + 精细拆词 + 朱拉隆功五声调音标**，直接生成成品。
- **方式 B（零 API 费用 · Google One Pro 网页版接力）**：不想额外使用 API Key 时，运行 5B 把字幕拆分成带专属 Prompt 的文本块存入 Google Drive，你复制到 **Google One Pro 自带的 Gemini Advanced** 对话框里跑完再存回即可。

In [ ]:
#@title 5.0 解析泰文 SRT & 用 PyThaiNLP 佛学词典预保护分词
from pythainlp.tokenize import word_tokenize
from pythainlp.corpus.common import thai_words
from pythainlp.util import dict_trie

PROTECTED_DHAMMA_WORDS = [
    "หลวงปู่ปราโมทย์", "หลวงพ่อปราโมทย์", "หลวงปู่ดูลย์", "หลวงพ่อพุธ",
    "ปาโมชฺโช", "วัดสวนสันติธรรม", "พุทธกาล", "สังสารวัฏ", "ธรรมาสน์",
    "สัมมาสมาธิ", "มิจฉาสมาธิ", "ไตรลักษณ์", "อริยสัจ", "อริยมรรคมีองค์ 8",
    "กุศลกรรมบถ", "สติปัฏฐาน", "แยกรูปแยกนาม", "รูปธรรม", "นามธรรม",
    "จิตผู้รู้", "จิตประภัสสร", "วิหารธรรม", "วิปัสสนา", "สมถะ",
    "วิมุตติ", "มรรคผล", "นิพพาน", "อวิชชา", "ตัณหา", "อุปาทาน",
    "โมหะ", "อโมหะ", "กิเลส", "อนุสติ", "อิริยาบถ", "ปฏิกูล",
    "ทอดกฐิน", "ผ้ากฐิน", "อนุโมทนา", "กรวดน้ำ", "ญาติโยม", "ครูบาอาจารย์",
    "ว่อกแว่ก", "เห็นอกเห็นใจ", "ส่อเสียด", "เพ้อเจ้อ", "เรี่ยไร",
    "เรียบง่าย", "เบื้องต้น", "ล้วนแต่", "มือถือ", "วันอาทิตย์", "ทำบุญ"
]
CUSTOM_DICT = dict_trie(set(thai_words()) | set(PROTECTED_DHAMMA_WORDS))

def parse_srt(srt_text: str):
    blocks = re.split(r'\r?\n\r?\n+', srt_text.strip())
    items = []
    pat = re.compile(r'(\d{2}):(\d{2}):(\d{2})[,.](\d{3})\s*-->\s*(\d{2}):(\d{2}):(\d{2})[,.](\d{3})')
    def to_sec(h, m, s, ms):
        return int(h)*3600 + int(m)*60 + int(s) + int(ms)/1000.0
    for b in blocks:
        lines = [l.strip() for l in b.splitlines() if l.strip()]
        m_idx, m_obj = -1, None
        for i, l in enumerate(lines[:3]):
            m = pat.search(l)
            if m:
                m_idx, m_obj = i, m
                break
        if m_obj is None:
            continue
        s_sec = to_sec(*m_obj.groups()[:4])
        e_sec = to_sec(*m_obj.groups()[4:])
        thai = " ".join(lines[m_idx+1:]).strip()
        thai = re.sub(r'<[^>]+>', '', thai).strip()
        if thai:
            tokens = [t.strip() for t in word_tokenize(thai, custom_dict=CUSTOM_DICT, engine="newmm") if t.strip()]
            items.append({
                "id": len(items) + 1,
                "start": round(s_sec, 2),
                "end": round(e_sec, 2),
                "thai": thai,
                "pre_tokens": tokens
            })
    return items

raw_items = parse_srt(raw_srt_file.read_text(encoding='utf-8'))
print(f"📖 已解析 {len(raw_items)} 句泰文字幕，示例前 3 句预分词：")
for x in raw_items[:3]:
    print(f"  [{x['id']}] {x['thai']} -> {x['pre_tokens']}")

In [ ]:
#@title 5A 全自动通道：调用 Gemini API 完成佛法校对 + 泰译中 + 精细拆词 + 朱拉五调音标
import time
from tqdm import tqdm
from google import genai
from google.genai import types

# 建议填入你自己的 Google AI Studio Key（或存入 Colab 左侧钥匙 Secrets 中命名为 GEMINI_API_KEY）
GEMINI_API_KEY = ""  #@param {type:"string"}
GEMINI_MODEL = "gemini-2.5-flash"  #@param ["gemini-2.5-flash", "gemini-2.5-pro"]
BATCH_SIZE = 30  #@param {type:"integer"}

if not GEMINI_API_KEY.strip():
    try:
        from google.colab import userdata
        GEMINI_API_KEY = userdata.get('GEMINI_API_KEY') or ""
    except Exception:
        pass

structured_ckpt_file = DIRS['manifests'] / f"{TASK_ID}_structured_ckpt.json"
done_map = {}
if structured_ckpt_file.exists():
    done_map = {x['id']: x for x in json.loads(structured_ckpt_file.read_text(encoding='utf-8'))}
    print(f"♻️ 已从断点恢复 {len(done_map)}/{len(raw_items)} 句")

SYSTEM_PROMPT = """你是一位精通南传上座部佛教（尤其是隆波帕默尊者 Wat Suan Santidham 禅修体系）与泰国朱拉隆功大学实用泰语音标（Chula Adapted IPA 五声调）的专业译经校对专家。

请对输入的泰语开示句子列表逐句完成以下 4 项工作，并严格输出 JSON 数组：
1. thai_clean: 若原文来自语音识别，结合隆波帕默尊者法义语境修正明显的同音错字（如 ภาวนา/สติ/สมาธิ/ไตรลักษณ์/หลวงปู่ดูลย์ 等），但保持尊者原话口吻，不要改写句意。
2. chinese: 贴近隆波帕默尊者慈悲平实、直指人心口吻的自然流畅简体中文翻译；核心佛学术语规范统一：
   - จิตผู้รู้=知者心, ไตรลักษณ์=三法印, วิหารธรรม=法住, แยกรูปแยกนาม=分离色法与名法, สังขาร=造作/行蕴, วิมุตติ=究竟解脱, กุศลกรรมบถ=十善业道, อริยสัจ 4=四圣谛。
3. words: 细颗粒度泰语词元数组（不可将多词粘成整句，不可从音节中间切碎元音），每个词包含：
   - thai: 泰语词元
   - romanization: 准确朱拉隆功 Adapted IPA 五声调音标（mid无标号, low=à, falling=â, high=á, rising=ǎ，长元音双写如 aa/ii/uu/ɛɛ/ɔɔ/əə，喉塞音 ʔ，送气音 pʰ/tʰ/kʰ/cʰ）
   - chinese: 精准词级中文释义（严禁使用“助词”等敷衍占位词，语气词须给出实际语气/语法义如“表感叹/语气柔化/了/呢”）
   - isCore: 布尔值，核心佛学词汇设为 true
"""

if not GEMINI_API_KEY.strip():
    print("ℹ️ 未填入 GEMINI_API_KEY。如你希望免费使用 Google One Pro 网页版 Gemini，请直接运行下方的【5B】单元生成接力提示包。")
else:
    client = genai.Client(api_key=GEMINI_API_KEY.strip())
    pending = [x for x in raw_items if x['id'] not in done_map]
    print(f"🚀 待处理 {len(pending)} 句，共 {(len(pending)+BATCH_SIZE-1)//BATCH_SIZE} 批...")

    for i in tqdm(range(0, len(pending), BATCH_SIZE)):
        batch = pending[i:i+BATCH_SIZE]
        payload = [{"id": x["id"], "thai": x["thai"], "hint_tokens": x["pre_tokens"]} for x in batch]
        user_msg = "请按系统要求处理以下泰语开示片段，返回包含字段 id, thai_clean, chinese, words 的 JSON 数组：\n" + json.dumps(payload, ensure_ascii=False)

        for attempt in range(3):
            try:
                resp = client.models.generate_content(
                    model=GEMINI_MODEL,
                    contents=user_msg,
                    config=types.GenerateContentConfig(
                        system_instruction=SYSTEM_PROMPT,
                        response_mime_type="application/json",
                        temperature=0.2,
                    ),
                )
                arr = json.loads(resp.text)
                if isinstance(arr, dict):
                    arr = arr.get("items") or arr.get("subtitles") or list(arr.values())[0]
                src_by_id = {x["id"]: x for x in batch}
                for row in arr:
                    rid = int(row["id"])
                    if rid in src_by_id:
                        src = src_by_id[rid]
                        words = []
                        for w_idx, w in enumerate(row.get("words") or []):
                            words.append({
                                "id": f"tok_{w_idx}",
                                "thai": str(w.get("thai", "")).strip(),
                                "romanization": str(w.get("romanization", "")).strip(),
                                "chinese": str(w.get("chinese", "")).strip(),
                                "isCore": bool(w.get("isCore", False))
                            })
                        done_map[rid] = {
                            "id": rid,
                            "start": src["start"],
                            "end": src["end"],
                            "thai": str(row.get("thai_clean") or src["thai"]).strip(),
                            "romanization": " ".join(w["romanization"] for w in words if w["romanization"]),
                            "chinese": str(row.get("chinese", "")).strip(),
                            "words": words
                        }
                ordered = [done_map[k] for k in sorted(done_map.keys())]
                structured_ckpt_file.write_text(json.dumps(ordered, ensure_ascii=False, indent=2), encoding='utf-8')
                break
            except Exception as e:
                print(f"⚠️ 第 {i//BATCH_SIZE + 1} 批重试({attempt+1}/3): {e}")
                time.sleep(3 * (attempt + 1))

    print(f"🎉 结构化翻译与标音全部完成！累计处理 {len(done_map)}/{len(raw_items)} 句。")

In [ ]:
#@title 5B 零费用接力模式（可选）：导出可直接粘贴给 Google One Pro 网页版 Gemini 的分段提示包
CHUNK_LINES = 120  #@param {type:"integer"}

AI_HEADER = """请把下面【泰语开示原始 SRT】转换为本网站专用【增强型泰汉逐词对齐 SRT】：
要求：
1. 保持原有编号与时间戳不变；
2. 修正明显语音识别错别字（尤其是 สติ/สมาธิ/ไตรลักษณ์/วิปัสสนา/หลวงปู่ปราโมทย์ 等佛学术语），但保留泰语口语原貌；
3. 每个时间段严格输出 4 行：
   - 第1行：泰文原文
   - 第2行：[GLOSS] 词1:释义1 | 词2:释义2 | ... （精细拆词，绝不用“助词”敷衍）
   - 第3行：[ROMAN] 整句朱拉隆功 Adapted IPA 五声调音标（mid无标号, low=à, falling=â, high=á, rising=ǎ）
   - 第4行：自然贴合隆波帕默尊者法义的简体中文翻译
4. 直接在代码块中输出完整 SRT 文本，不要省略任何一句。

--- 以下是待处理字幕 ---
"""

chunk_paths = []
for start in range(0, len(raw_items), CHUNK_LINES):
    sub = raw_items[start:start+CHUNK_LINES]
    body = []
    for it in sub:
        body.append(f"{it['id']}\n{fmt_srt_time(it['start'])} --> {fmt_srt_time(it['end'])}\n{it['thai']}\n")
    out_txt = DIRS['chunks'] / f"{TASK_ID}_chunk_{start//CHUNK_LINES + 1:02d}.txt"
    out_txt.write_text(AI_HEADER + "\n".join(body), encoding="utf-8")
    chunk_paths.append(out_txt)

print(f"📦 已生成 {len(chunk_paths)} 份网页端 Gemini 提示包，保存在：{DIRS['chunks']}")

## 6️⃣ 一键导出：增强型 SRT（导入网站）、结构化 JSON、NotebookLM Markdown

In [ ]:
#@title 6.1 导出 3 种成品到 Google Drive 并更新 manifest.json
def fmt_mm_ss(sec: float) -> str:
    sec_i = int(round(max(0.0, sec)))
    h, rem = divmod(sec_i, 3600)
    m, s = divmod(rem, 60)
    return f"{h:02d}:{m:02d}:{s:02d}" if h > 0 else f"{m:02d}:{s:02d}"

if not structured_ckpt_file.exists():
    print("⚠️ 未检测到 5A 结构化结果，将使用基础泰文 SRT 生成纯泰语归档。")
    final_items = [{
        "id": x["id"],
        "start": x["start"],
        "end": x["end"],
        "thai": x["thai"],
        "romanization": "",
        "chinese": "",
        "words": [{"id": f"tok_{j}", "thai": t, "romanization": "", "chinese": ""} for j, t in enumerate(x["pre_tokens"])]
    } for x in raw_items]
else:
    final_items = json.loads(structured_ckpt_file.read_text(encoding='utf-8'))

# 1) 导出【结构化 JSON】：直接兼容本站 SubtitleLine[] 与全库检索
subtitles_payload = []
for item in final_items:
    subtitles_payload.append({
        "id": item["id"],
        "timeStart": fmt_mm_ss(item["start"]),
        "timeEnd": fmt_mm_ss(item["end"]),
        "secondsStart": int(round(item["start"])),
        "secondsEnd": int(round(item["end"])),
        "thai": item["thai"],
        "romanization": item["romanization"],
        "chinese": item["chinese"],
        "words": item["words"]
    })

json_export = {
    "date": TASK_ID,
    "thaiTitle": TALK_TITLE_TH,
    "chineseTitle": TALK_TITLE_ZH,
    "sourceMp3": DHAMMA_MP3_URL,
    "sourceYouTube": YOUTUBE_URL,
    "totalSentences": len(subtitles_payload),
    "subtitles": subtitles_payload
}
json_path = DIRS['json'] / f"{TASK_ID}.taixue.json"
json_path.write_text(json.dumps(json_export, ensure_ascii=False, indent=2), encoding="utf-8")

# 2) 导出【增强型 SRT-GLOSS】：可直接在网站点「上传/导入字幕」拖拽加载
esrt_blocks = []
for sub, raw in zip(subtitles_payload, final_items):
    gloss = " | ".join(f"{w['thai']}:{w['chinese']}" for w in sub["words"] if w.get("thai"))
    lines = [
        str(sub["id"]),
        f"{fmt_srt_time(raw['start'])} --> {fmt_srt_time(raw['end'])}",
        sub["thai"]
    ]
    if gloss:
        lines.append(f"[GLOSS] {gloss}")
    if sub["romanization"]:
        lines.append(f"[ROMAN] {sub['romanization']}")
    if sub["chinese"]:
        lines.append(sub["chinese"])
    esrt_blocks.append("\n".join(lines) + "\n")

esrt_path = DIRS['enhanced_srt'] / f"{TASK_ID}_enhanced.srt"
esrt_path.write_text("\n".join(esrt_blocks), encoding="utf-8")

# 3) 导出【NotebookLM 知识库 Markdown】：泰汉对照逐段实录
md_lines = [
    f"# {TALK_TITLE_ZH}",
    f"**泰语原题**: {TALK_TITLE_TH}",
    f"**日期**: {TASK_ID} | **总句数**: {len(subtitles_payload)}",
    "",
    "---",
    ""
]
for sub in subtitles_payload:
    md_lines.append(f"- **[{sub['timeStart']}]** {sub['thai']}")
    if sub["chinese"]:
        md_lines.append(f"  - {sub['chinese']}")

md_path = DIRS['markdown'] / f"{TASK_ID}_transcript.md"
md_path.write_text("\n".join(md_lines), encoding="utf-8")

# 更新进度清单 manifest.json
manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
manifest[TASK_ID] = {
    "date": TASK_ID,
    "title_th": TALK_TITLE_TH,
    "title_zh": TALK_TITLE_ZH,
    "sentences": len(subtitles_payload),
    "official_cc": official_cc_found,
    "json": str(json_path),
    "enhanced_srt": str(esrt_path),
    "markdown": str(md_path)
}
MANIFEST_PATH.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")

print("✅ 全部三大成品已生成并归档至 Google Drive：")
print("  1. 结构化 JSON :", json_path)
print("  2. 增强型 SRT  :", esrt_path, "（可直接拖入网站导入！）")
print("  3. Markdown    :", md_path, "（可直接喂给 NotebookLM！）")